# VK CaseChamp 2026 — очистка, воронка, драйверы, сегменты

In [4]:
import pandas as pd, numpy as np, warnings
from scipy import stats
from statsmodels.stats.contingency_tables import StratifiedTable
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore'); pd.set_option('display.width',200, 'display.max_columns',40)
SRC='../data/Датасет.xlsx'; OUT='../output/Бизнесы_очищенные.xlsx'
xl=pd.ExcelFile(SRC)
pd.DataFrame([(s,*pd.read_excel(SRC,sheet_name=s).shape) for s in xl.sheet_names],columns=['лист','строк','столбцов'])

,лист,строк,столбцов
0,Бизнесы,9754,28
1,Маркет Вконтакте — итоги,26,4
2,Маркет Вконтакте — по дням,30,19
3,Маркет Вконтакте — источники,7,5
4,Маркет Вконтакте — категории,18,5
5,Покупатели,15,6
6,Покупатели по регионам,89,5


## 1. Что в листах
Короткий обзор: первые строки каждого листа, кроме «Бизнесы» (его разбираем ниже).

In [5]:
for s in xl.sheet_names[1:]:
    d=pd.read_excel(SRC,sheet_name=s); print(f'\n### {s} {d.shape}'); display(d.head(4))


### Маркет Вконтакте — итоги (26, 4)


,Метрика,Маркет Вконтакте,Партнёр 1,Всего
0,Показы товаров,4777750023,5729628790,NaN
1,Открытия карточки товара,193512461,86882822,NaN
2,Нажатия на кнопку в карточке,5969063,36626075,NaN
3,Прямые переходы к Партнёру 1 (без открытия кар...,—,106914679,NaN



### Маркет Вконтакте — по дням (30, 19)


,Дата,DAU Маркета Вконтакте,Видели товары (уник.),Открывали карточку (уник.),Поисковые запросы,Поисковые запросы с открытием карточки,Показы товаров: Маркет Вконтакте,Показы товаров: Партнёр 1,Открытия карточки: Маркет Вконтакте,Открытия карточки: Партнёр 1,Нажатия на кнопку: Маркет Вконтакте,Нажатия на кнопку: Партнёр 1,Заказы: Маркет Вконтакте,Заказы: Партнёр 1,Оплаченные заказы: Маркет Вконтакте,"GMV, ₽: Маркет Вконтакте","GMV, ₽: Партнёр 1",Покупатели: Маркет Вконтакте,Покупатели: Партнёр 1
0,2026-09-01,128656,22957807,2320353,1076659,204671,164388078,150004733,6030287,2190784,234998,893153,20370,128947,159,66555585,113125610,17959,105562
1,2026-09-02,121217,23147856,2422557,923572,160155,160332304,161037443,5985707,2414712,192427,978572,12687,157446,209,49599901,131042721,10900,128206
2,2026-09-03,125152,23515925,2591746,937298,164719,163016247,171994301,6178847,2665858,196286,1069523,13428,167541,171,56951176,138066685,11571,137502
3,2026-09-04,127084,23550688,2710678,978351,178502,168725840,191797622,6417539,2892153,212677,1210098,15366,168546,185,52908793,141976696,13468,136777



### Маркет Вконтакте — источники (7, 5)


,Источник,Маркет Вконтакте,"Доля, %: Маркет Вконтакте",Партнёр 1,"Доля, %: Партнёр 1"
0,Другое,77753037.0,40.2,5380616,6.2
1,Витрина магазина,75604323.0,39.1,714540,0.8
2,Лента,18795696.0,9.7,15622386,18.0
3,Клипы,17239066.0,8.9,64917109,74.7


... (2 outputs left out)


### Покупатели (15, 6)


,Пол,Возраст,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1
0,Ж,35–44,66329.0,91318.0,700024,1113644
1,Ж,25–34,54635.0,75822.0,363334,574071
2,Ж,45–54,37784.0,53843.0,520765,840735
3,Ж,18–24,31697.0,44398.0,111820,168479



### Покупатели по регионам (89, 5)


,Регион,Покупатели: Маркет Вконтакте,Заказы: Маркет Вконтакте,Покупатели: Партнёр 1,Заказы: Партнёр 1
0,г. Москва,30488.0,42652.0,146366,229256
1,г. Санкт-Петербург,24687.0,36107.0,155546,248956
2,Челябинская область,20465.0,28558.0,67879,107682
3,Пермский край,13366.0,19489.0,54523,85703


In [6]:
raw=pd.read_excel(SRC,sheet_name='Бизнесы')
short=['id','topic','type','subs','posts','clips','pposts','pclips','v','l','r','pv','pl','pr','card','loaded','active',
       'checkout','deliv','pay','orig','ref','orders','gmv','aov','ads','services','price']
COLS=dict(zip(short,raw.columns)); b=raw.copy(); b.columns=short
pd.DataFrame({'код':short,'исходное название':raw.columns,'пропусков':b.isna().sum().values})

,код,исходное название,пропусков
0,id,ID бизнеса,0
1,topic,Тематика,0
2,type,Тип бизнеса (товары / услуги),0
3,subs,Число подписчиков,0
4,posts,Посты за 30 дней,0
5,clips,Клипы за 30 дней,0
6,pposts,Посты с товаром за 30 дней,0
7,pclips,Клипы с товаром за 30 дней,0
8,v,Просмотры обычного контента,0
9,l,Лайки обычного контента,0


## 2. Очистка
Лог: шаг, правило, число строк, что делаем.

In [7]:
LOG=[]
def log(step,rule,n,action): LOG.append(dict(шаг=step,правило=rule,строк=int(n),действие=action))
log('0','Исходный лист «Бизнесы»',len(b),'—')

# 2.1 Область кейса — только товары
b['in_scope']=(b.type!='услужное')
log('1','Тип = «услужное» (кейс про товары)',(~b.in_scope).sum(),'исключены из анализа, оставлены в файле с in_scope=0')

# 2.2 Флаги противоречий
F={}
F['f_dup']=('Полный дубль строки (кроме ID)', b.drop(columns='id').duplicated(keep=False))
o=b.orders>0
F['f_aov_mismatch']=('Средний чек ≠ GMV ÷ заказы (расхождение >5%)', o&(b.gmv>0)&((b.aov/(b.gmv/b.orders.where(o))-1).abs()>0.05))
F['f_orders_no_gmv']=('Заказы > 0, но GMV = 0', o&(b.gmv==0))
F['f_orders_no_checkout']=('Заказы через чекаут без подключённого чекаута', o&(b.checkout==0))
F['f_orders_no_catalog']=('Заказы без активных товаров в каталоге', o&(b.active==0))
F['f_pv_no_pposts']=('Просмотры товарного контента без товарных постов/клипов', (b.pv>0)&(b.pposts+b.pclips==0))
F['f_card_no_pv']=('Открытия карточки из контента без просмотров товарного контента', (b.card>0)&(b.pv==0))
F['f_pposts_gt_posts']=('Товарных постов больше, чем всех постов', b.pposts>b.posts)
F['f_pv_gt_v']=('Просмотры товарного > обычного контента (проверка, не ошибка)', b.pv>b.v)
F['f_likes_gt_views']=('Лайков больше, чем просмотров', b.l>b.v)
F['f_deliv_no_checkout']=('Доставка подключена без чекаута', (b.deliv==1)&(b.checkout==0))
F['f_pay_no_checkout']=('Оплата подключена без чекаута', (b.pay==1)&(b.checkout==0))
F['f_card_no_loaded']=('Открытия карточки при 0 загруженных товаров', (b.card>0)&(b.loaded==0))
p99=b.ads.quantile(.99)
F['f_ads_outlier']=(f'Реклама выше 99-го перцентиля ({p99:,.0f} ₽)', b.ads>p99)
F['f_price_outlier']=('Медианная цена = 0 или > 200 тыс. ₽', (b.price==0)|(b.price>2e5))
F['f_orig_na']=('Нет оценки оригинальности контента', b.orig.isna())
for k,(rule,mask) in F.items():
    b[k]=mask.astype(int); log('2',rule,(mask&b.in_scope).sum(),'флаг '+k+' (в товарной области)')

# 2.3 Исправления
b['aov_calc']=np.where(o&(b.gmv>0), b.gmv/b.orders.where(o), np.nan)
log('3','Средний чек пересчитан как GMV ÷ заказы',b.aov_calc.notna().sum(),'новый столбец aov_calc, исходный aov не используем')
b['ads_w']=b.ads.clip(upper=p99); log('3','Реклама винзоризирована по 99-му перцентилю',(b.ads>p99).sum(),'столбец ads_w для средних/сумм')

# 2.4 Аналитическая выборка: в области кейса и без критичных противоречий по заказам
crit=b[['f_orders_no_gmv','f_orders_no_checkout','f_orders_no_catalog']].any(axis=1)
b['analysis']=(b.in_scope & ~crit).astype(int)
log('4','Критичные противоречия по заказам (без GMV / чекаута / каталога)',(b.in_scope&crit).sum(),'исключены из воронки, драйверов и сегментов')
log('5','Итог: аналитическая выборка (товары)',b.analysis.sum(),'—')
LOGDF=pd.DataFrame(LOG); LOGDF

,шаг,правило,строк,действие
0,0,Исходный лист «Бизнесы»,9754,—
1,1,Тип = «услужное» (кейс про товары),2428,"исключены из анализа, оставлены в файле с in_s..."
2,2,Полный дубль строки (кроме ID),42,флаг f_dup (в товарной области)
3,2,Средний чек ≠ GMV ÷ заказы (расхождение >5%),212,флаг f_aov_mismatch (в товарной области)
4,2,"Заказы > 0, но GMV = 0",7,флаг f_orders_no_gmv (в товарной области)
5,2,Заказы через чекаут без подключённого чекаута,14,флаг f_orders_no_checkout (в товарной области)
6,2,Заказы без активных товаров в каталоге,10,флаг f_orders_no_catalog (в товарной области)
7,2,Просмотры товарного контента без товарных пост...,906,флаг f_pv_no_pposts (в товарной области)
8,2,Открытия карточки из контента без просмотров т...,27,флаг f_card_no_pv (в товарной области)
9,2,"Товарных постов больше, чем всех постов",10,флаг f_pposts_gt_posts (в товарной области)


**Пояснения к флагам.**
- Средний чек расходится с GMV÷заказы у большинства продавцов, причём кластерами (отношение ≈0,85 и ≈1,13) — похоже на разные базы расчёта (возвраты/доставка/скидки). Вопрос на Q&A; в расчётах используем `aov_calc`.
- Просмотры товарного контента без товарных постов: вероятно, товар привязан к старым публикациям или репостам (окно 30 дней). Не исключаем, но в драйверах «товарный контент» = есть товарные посты/клипы.
- Реклама: единичные значения до 17 млн ₽ — сохраняем для флага «есть реклама», в суммах используем `ads_w`.

In [8]:
A=b[b.analysis==1].copy()
A['sell']=(A.orders>0).astype(int)
A['size']=pd.cut(A.subs,[-1,300,1000,5000,1e12],labels=['<300','300–1k','1k–5k','5k+'])
top=A.topic.value_counts().index[:8]; A['tgroup']=np.where(A.topic.isin(top),A.topic,'Прочие тематики')
print('аналитическая выборка',len(A),'| с ≥1 заказом',A.sell.sum(),f'({A.sell.mean():.1%})')

аналитическая выборка 7303 | с ≥1 заказом 272 (3.7%)


## 3. Воронка бизнеса
Строгая (накопительная) воронка: каждый следующий шаг — подмножество предыдущего.

In [9]:
steps=[('Товарные бизнесы',np.ones(len(A),bool)),
       ('Есть каталог (активные товары)',A.active>0),
       ('+ чекаут',A.checkout==1),('+ доставка',A.deliv==1),('+ оплата',A.pay==1)]
rows=[]; m=np.ones(len(A),bool)
for name,cond in steps:
    m=m&np.asarray(cond); rows.append((name,m.sum(),A.sell[m].sum()))
rows.append(('≥1 заказ (среди полного стека)',A.sell[m].sum(),A.sell[m].sum()))
FUN=pd.DataFrame(rows,columns=['шаг','бизнесов','из них с заказом'])
FUN['конверсия шага']=(FUN.бизнесов/FUN.бизнесов.shift()).round(3); FUN['доля от старта']=(FUN.бизнесов/FUN.бизнесов[0]).round(3)
FUN['доля продающих на шаге']=(FUN['из них с заказом']/FUN.бизнесов).round(3); FUN

,шаг,бизнесов,из них с заказом,конверсия шага,доля от старта,доля продающих на шаге
0,Товарные бизнесы,7303,272,NaN,1.000,0.037
1,Есть каталог (активные товары),3948,272,0.541,0.541,0.069
2,+ чекаут,3635,272,0.921,0.498,0.075
3,+ доставка,719,123,0.198,0.098,0.171
4,+ оплата,94,16,0.131,0.013,0.170
5,≥1 заказ (среди полного стека),16,16,0.170,0.002,1.000


Альтернативный разрез: **на каком уровне инструментов бизнес остановился** (взаимоисключающие состояния) и доля продающих на каждом.

In [10]:
def state(r):
    if r.active==0: return '0 Без каталога'
    if r.checkout==0: return '1 Каталог без чекаута'
    if r.deliv==0 and r.pay==0: return '2 Чекаут, без доставки и оплаты'
    if r.deliv==1 and r.pay==0: return '3 Чекаут + доставка'
    if r.deliv==0 and r.pay==1: return '3b Чекаут + оплата'
    return '4 Полный стек'
A['stack']=A.apply(state,axis=1)
ST=A.groupby('stack').agg(бизнесов=('id','size'),продающих=('sell','sum'),доля_продающих=('sell','mean'),
   заказов=('orders','sum'),GMV=('gmv','sum'),медиана_подписчиков=('subs','median')).round(3)
ST['доля_бизнесов']=(ST.бизнесов/len(A)).round(3); ST['доля_заказов']=(ST.заказов/A.orders.sum()).round(3); ST

,бизнесов,продающих,доля_продающих,заказов,GMV,медиана_подписчиков,доля_бизнесов,доля_заказов
stack,,,,,,,,
0 Без каталога,3355,0,0.000,0,0,410.0,0.459,0.000
1 Каталог без чекаута,313,0,0.000,0,0,1300.0,0.043,0.000
"2 Чекаут, без доставки и оплаты",2810,139,0.049,1778,4102849,640.0,0.385,0.521
3 Чекаут + доставка,625,107,0.171,1336,4420108,1000.0,0.086,0.391
3b Чекаут + оплата,106,10,0.094,32,131490,1750.0,0.015,0.009
4 Полный стек,94,16,0.170,267,848840,1250.0,0.013,0.078


## 4. Драйверы продаж в сопоставимых группах
База: товарные бизнесы с каталогом. Сравнение «есть признак / нет» внутри страт **размер аудитории (4) × тематика (8 крупнейших + прочие)** — тест Кохрана–Мантеля–Хензеля, общий OR и разница долей Мантеля–Хензеля (п.п.), поправка Холма на множественные сравнения.
Сначала — проверка сопоставимости: у кого есть признак, тот обычно крупнее, поэтому сырые сравнения завышают эффект.

In [11]:
K=A[A.active>0].copy(); K['strat']=K['size'].astype(str)+' | '+K.tgroup
K['Товарный контент (посты/клипы с товаром)']=((K.pposts+K.pclips)>0).astype(int)
K['Клипы (любые)']=(K.clips>0).astype(int)
K['Товарные клипы']=(K.pclips>0).astype(int)
K['Активный контент (любой пост/клип)']=((K.posts+K.clips)>0).astype(int)
K['Реферальная программа']=K.ref
K['Реклама (>0 ₽)']=(K.ads>0).astype(int)
K['Каталог ≥ 20 товаров']=(K.active>=20).astype(int)
K['Чекаут']=K.checkout; K['Доставка']=K.deliv; K['Онлайн-оплата']=K.pay
K['Оригинальный контент']=K.orig
DR=[c for c in K.columns if c[0].isupper() and c not in ('Оригинальный контент',)]+['Оригинальный контент']
def mh(df,flag):
    df=df[df[flag].notna()]; tabs=[]; num=den=0
    for _,g in df.groupby('strat'):
        t=pd.crosstab(g[flag],g.sell).reindex(index=[1,0],columns=[1,0],fill_value=0).values
        if (t.sum(1)==0).any() or (t.sum(0)==0).any(): continue
        tabs.append(t); n1,n0,N=t[0].sum(),t[1].sum(),t.sum()
        num+=(t[0,0]*n0-t[1,0]*n1)/N; den+=n1*n0/N
    st=StratifiedTable(tabs); lo,hi=st.oddsratio_pooled_confint()
    g1,g0=df[df[flag]==1],df[df[flag]==0]
    return dict(признак=flag,есть=len(g1),нет=len(g0),
        **{'продают, есть':g1.sell.mean(),'продают, нет':g0.sell.mean(),'медиана подп., есть':g1.subs.median(),'медиана подп., нет':g0.subs.median(),
        'OR (MH)':st.oddsratio_pooled,'OR низ':lo,'OR верх':hi,'разница долей MH, п.п.':100*num/den,'p':st.test_null_odds(correction=True).pvalue,'страт':len(tabs)})
D=pd.DataFrame([mh(K,f) for f in DR])
D['p Холм']=multipletests(D.p,method='holm')[1]
D['вывод']=np.select([(D['p Холм']<.05)&(D['OR (MH)']>1),(D['p Холм']<.05)&(D['OR (MH)']<1),D.p<.05],['драйвер','обратная связь','сигнал (не выдержал поправку)'],'не подтверждено')
D=D.sort_values('OR (MH)',ascending=False).round(4); D

,признак,есть,нет,"продают, есть","продают, нет","медиана подп., есть","медиана подп., нет",OR (MH),OR низ,OR верх,"разница долей MH, п.п.",p,страт,p Холм,вывод
7,Чекаут,3635,313,0.0748,0.0000,730.0,1300.0,inf,NaN,NaN,11.3495,0.0000,32,0.0000,драйвер
6,Каталог ≥ 20 товаров,1518,2430,0.1357,0.0272,1150.0,600.0,4.0960,3.0052,5.5827,8.5965,0.0000,32,0.0000,драйвер
2,Товарные клипы,86,3862,0.2326,0.0653,945.0,770.0,2.7240,1.5092,4.9165,12.5465,0.0004,24,0.0023,драйвер
8,Доставка,806,3142,0.1526,0.0474,1050.0,710.0,2.5535,1.9495,3.3447,8.2258,0.0000,32,0.0000,драйвер
0,Товарный контент (посты/клипы с товаром),246,3702,0.1707,0.0621,1050.0,760.0,2.1554,1.4500,3.2040,7.8959,0.0001,31,0.0006,драйвер
10,Оригинальный контент,951,2021,0.1020,0.0698,530.0,1000.0,2.0106,1.4639,2.7615,5.0975,0.0000,32,0.0001,драйвер
4,Реферальная программа,102,3846,0.1667,0.0663,1850.0,760.0,1.6341,0.9152,2.9179,5.5867,0.1188,22,0.2376,не подтверждено
3,Активный контент (любой пост/клип),2989,959,0.0773,0.0428,840.0,600.0,1.5526,1.0880,2.2156,2.2981,0.0180,32,0.0899,сигнал (не выдержал поправку)
5,Реклама (>0 ₽),262,3686,0.1450,0.0635,2600.0,720.0,1.5085,1.0161,2.2395,4.2102,0.0472,29,0.1417,сигнал (не выдержал поправку)
9,Онлайн-оплата,211,3737,0.1232,0.0658,1400.0,730.0,1.4542,0.9252,2.2855,3.5767,0.1228,32,0.2376,не подтверждено


Размер каталога — дозозависимость внутри размеров аудитории:

In [12]:
K['catalog_bin']=pd.cut(K.active,[0,5,20,100,1e9],labels=['1–5','6–20','21–100','100+'])
CAT=K.pivot_table(index='catalog_bin',columns='size',values='sell',aggfunc='mean',observed=True).round(3)
CAT['все']=K.groupby('catalog_bin',observed=True).sell.mean().round(3); CAT['n']=K.catalog_bin.value_counts(); CAT

size,<300,300–1k,1k–5k,5k+,все,n
catalog_bin,,,,,,
1–5,0.012,0.011,0.019,0.038,0.016,1031
6–20,0.026,0.022,0.039,0.099,0.036,1438
21–100,0.022,0.048,0.149,0.218,0.097,961
100+,0.058,0.105,0.191,0.409,0.214,518


## 5. Сегментация (правила, взаимоисключающие, по приоритету)
1. **Полный стек** — каталог + чекаут + доставка + оплата.
2. **Продаёт через контент** — каталог + товарные посты/клипы.
3. **Витрина с живым сообществом** — каталог, нет товарного контента, но есть обычные посты/клипы.
4. **Заброшенная витрина** — каталог, нет публикаций за 30 дней.
5. **Контент без витрины (продажи в переписке/вне VK)** — нет каталога, есть публикации.
6. **Спящие** — нет каталога и публикаций.

In [13]:
def seg(r):
    hasc=r.active>0; pc=(r.pposts+r.pclips)>0; act=(r.posts+r.clips)>0
    if hasc and r.checkout and r.deliv and r.pay: return '1 Полный стек'
    if hasc and pc: return '2 Продаёт через контент'
    if hasc and act: return '3 Витрина с живым сообществом'
    if hasc: return '4 Заброшенная витрина'
    if act: return '5 Контент без витрины'
    return '6 Спящие'
A['segment']=A.apply(seg,axis=1)
SG=A.groupby('segment').agg(бизнесов=('id','size'),медиана_подписчиков=('subs','median'),медиана_товаров=('active','median'),
   доля_с_доставкой=('deliv','mean'),продающих=('sell','sum'),доля_продающих=('sell','mean'),
   заказов=('orders','sum'),GMV=('gmv','sum'))
SG['доля_бизнесов']=SG.бизнесов/len(A); SG['доля_заказов']=SG.заказов/A.orders.sum()
SG['заказов_на_бизнес']=SG.заказов/SG.бизнесов
SG['сообществ_1k+']=A[A.subs>=1000].groupby('segment').size()
SG=SG.round(3); SG

,бизнесов,медиана_подписчиков,медиана_товаров,доля_с_доставкой,продающих,доля_продающих,заказов,GMV,доля_бизнесов,доля_заказов,заказов_на_бизнес,сообществ_1k+
segment,,,,,,,,,,,,
1 Полный стек,94,1250.0,59.0,1.000,16,0.170,267,848840,0.013,0.078,2.840,49
2 Продаёт через контент,231,1100.0,31.0,0.424,36,0.156,481,1846010,0.032,0.141,2.082,122
3 Витрина с живым сообществом,2693,820.0,11.0,0.174,182,0.068,2561,6524704,0.369,0.750,0.951,1216
4 Заброшенная витрина,930,600.0,11.0,0.157,38,0.041,104,283733,0.127,0.030,0.112,347
5 Контент без витрины,2592,430.0,0.0,0.011,0,0.000,0,0,0.355,0.000,0.000,871
6 Спящие,763,380.0,0.0,0.012,0,0.000,0,0,0.104,0.000,0.000,208


Сопоставимость сегментов: доля продающих внутри одинакового размера аудитории.

In [14]:
SS=A.pivot_table(index='segment',columns='size',values='sell',aggfunc='mean',observed=True).round(3); SS

size,<300,300–1k,1k–5k,5k+
segment,,,,
1 Полный стек,0.130,0.043,0.167,0.389
2 Продаёт через контент,0.048,0.094,0.169,0.316
3 Витрина с живым сообществом,0.018,0.032,0.088,0.201
4 Заброшенная витрина,0.022,0.025,0.056,0.141
5 Контент без витрины,0.000,0.000,0.000,0.000
6 Спящие,0.000,0.000,0.000,0.000


## 6. Пять ключевых гипотез
- **H1** Доставка — главный барьер к первой продаже (вопрос 5).
- **H2** Каталог от 20 товаров — порог, после которого бизнес продаёт (вопрос 2).
- **H3** Товарный контент (шопсы, товарные клипы) повышает продажи (вопрос 3).
- **H4** Маркет как витрина не продаёт: покупка начинается с контента (вопрос 4).
- **H5** Эффект инструментов сильнее у сообществ от 1 тыс. подписчиков → они целевые (вопрос 2).

In [15]:
H=[]
for h,f in [('H1','Доставка'),('H2','Каталог ≥ 20 товаров'),('H3a','Товарный контент (посты/клипы с товаром)'),('H3b','Товарные клипы')]:
    r=D.set_index('признак').loc[f]; H.append(dict(гипотеза=h,признак=f,**{k:r[k] for k in ['есть','продают, есть','продают, нет','OR (MH)','OR низ','OR верх','разница долей MH, п.п.','p Холм']}))
H=pd.DataFrame(H).round(4); H

,гипотеза,признак,есть,"продают, есть","продают, нет",OR (MH),OR низ,OR верх,"разница долей MH, п.п.",p Холм
0,H1,Доставка,806,0.1526,0.0474,2.5535,1.9495,3.3447,8.2258,0.0000
1,H2,Каталог ≥ 20 товаров,1518,0.1357,0.0272,4.0960,3.0052,5.5827,8.5965,0.0000
2,H3a,Товарный контент (посты/клипы с товаром),246,0.1707,0.0621,2.1554,1.4500,3.2040,7.8959,0.0006
3,H3b,Товарные клипы,86,0.2326,0.0653,2.7240,1.5092,4.9165,12.5465,0.0023


H1 — масштаб рычага: сколько сообществ застряли на «чекаут без доставки» и сколько продающих могло бы добавиться при эффекте из сопоставимых групп (оценка, не прогноз: причинность не доказана).

In [16]:
stuck=(A['stack']=='2 Чекаут, без доставки и оплаты').sum(); rd=D.set_index('признак').loc['Доставка','разница долей MH, п.п.']/100
print(f'застряли без доставки: {stuck}; +{rd*100:.1f} п.п. → ≈{stuck*rd:.0f} доп. продающих к {A.sell.sum()} текущим (+{stuck*rd/A.sell.sum():.0%})')

застряли без доставки: 2810; +8.2 п.п. → ≈231 доп. продающих к 272 текущим (+85%)


H4 — Маркет против Партнёра 1 (листы «итоги», «источники», «категории»).

In [17]:
it=pd.read_excel(SRC,sheet_name='Маркет Вконтакте — итоги').set_index('Метрика')
num=lambda m,c: float(it.loc[m,c])
M=pd.DataFrame({ch:{'открытия карточки':num('Открытия карточки товара',ch),'заказы':num('Оформленные заказы',ch),'GMV, ₽':num('GMV, ₽',ch)} for ch in ['Маркет Вконтакте','Партнёр 1']}).T
M['CR карточка→заказ, %']=100*M['заказы']/M['открытия карточки']
print('оплачено внутри VK, % заказов Маркета:',round(100*num('Оплаченные заказы','Маркет Вконтакте')/num('Оформленные заказы','Маркет Вконтакте'),1))
display(M.round(2))
src=pd.read_excel(SRC,sheet_name='Маркет Вконтакте — источники'); display(src)
cat=pd.read_excel(SRC,sheet_name='Маркет Вконтакте — категории')
cat=cat[~cat['Категория'].isin(['Другое','Недвижимость','Все товары','Не определена'])].copy()   # нет сопоставимых данных у одного из каналов
cat['CR Маркет, %']=100*cat.iloc[:,2]/cat.iloc[:,1]; cat['CR Партнёр 1, %']=100*cat.iloc[:,4]/cat.iloc[:,3]
print('категорий, где CR Партнёра 1 выше:',(cat['CR Партнёр 1, %']>cat['CR Маркет, %']).sum(),'из',len(cat))
CATM=cat[['Категория','CR Маркет, %','CR Партнёр 1, %']].round(3); CATM

оплачено внутри VK, % заказов Маркета: 1.2


,открытия карточки,заказы,"GMV, ₽","CR карточка→заказ, %"
Маркет Вконтакте,193512461.0,413740.0,1.564547e+09,0.21
Партнёр 1,86882822.0,4927525.0,5.119515e+09,5.67


,Источник,Маркет Вконтакте,"Доля, %: Маркет Вконтакте",Партнёр 1,"Доля, %: Партнёр 1"
0,Другое,77753037.0,40.2,5380616,6.2
1,Витрина магазина,75604323.0,39.1,714540,0.8
2,Лента,18795696.0,9.7,15622386,18.0
3,Клипы,17239066.0,8.9,64917109,74.7
4,Поиск,3707521.0,1.9,159788,0.2
5,Главная Маркета Вконтакте,412818.0,0.2,88383,0.1
6,в т.ч. из реферальных постов и клипов,NaN,NaN,82324081,94.8


категорий, где CR Партнёра 1 выше: 14 из 14


,Категория,"CR Маркет, %","CR Партнёр 1, %"
0,Гардероб,0.088,2.879
1,Хобби и развлечения,0.345,4.728
3,Дом и дача,0.138,5.764
4,Продукты питания,0.958,6.854
5,Красота и здоровье,0.115,8.788
6,Транспорт,0.096,5.789
7,Электроника,0.078,1.099
8,Спорт и отдых,0.138,4.387
9,Детские товары,0.217,4.148
10,Ремонт и строительство,0.038,2.003


H5 — меняется ли эффект инструмента с размером аудитории. OR по каждой страте размера (внутри — по тематикам) и тест Бреслоу–Дэя на однородность OR. Плюс сравнение «есть/нет» по абсолютной разнице долей: эффект в п.п. важнее для числа продающих.

In [18]:
rows=[]
for f in ['Доставка','Каталог ≥ 20 товаров','Товарный контент (посты/клипы с товаром)']:
    tabs=[]
    for s,g in K.groupby('size',observed=True):
        t=pd.crosstab(g[f],g.sell).reindex(index=[1,0],columns=[1,0],fill_value=0).values
        p1=g[g[f]==1].sell.mean(); p0=g[g[f]==0].sell.mean()
        rows.append(dict(признак=f,аудитория=str(s),n_есть=int((g[f]==1).sum()),**{'продают, есть':p1,'продают, нет':p0,'разница, п.п.':100*(p1-p0),
                    'OR':(t[0,0]+.5)*(t[1,1]+.5)/((t[0,1]+.5)*(t[1,0]+.5))}))
        tabs.append(t+0.5)
    bd=StratifiedTable(tabs).test_equal_odds(); rows.append(dict(признак=f,аудитория='Бреслоу–Дэй p',OR=bd.pvalue))
H5=pd.DataFrame(rows).round(3); H5

,признак,аудитория,n_есть,"продают, есть","продают, нет","разница, п.п.",OR
0,Доставка,<300,195.0,0.062,0.015,4.617,4.234
1,Доставка,300–1k,208.0,0.082,0.022,5.981,3.969
2,Доставка,1k–5k,242.0,0.174,0.064,10.959,3.078
3,Доставка,5k+,161.0,0.323,0.161,16.216,2.484
4,Доставка,Бреслоу–Дэй p,NaN,NaN,NaN,NaN,0.534
5,Каталог ≥ 20 товаров,<300,369.0,0.030,0.020,1.020,1.559
6,Каталог ≥ 20 товаров,300–1k,364.0,0.066,0.017,4.939,4.095
7,Каталог ≥ 20 товаров,1k–5k,497.0,0.161,0.032,12.875,5.669
8,Каталог ≥ 20 товаров,5k+,288.0,0.316,0.074,24.155,5.602
9,Каталог ≥ 20 товаров,Бреслоу–Дэй p,NaN,NaN,NaN,NaN,0.020


In [19]:
big=K.subs>=1000
print('доля сообществ 1k+ среди товарных с каталогом:',f'{big.mean():.0%}','| доля продающих 1k+ vs <1k:',f'{K[big].sell.mean():.1%} vs {K[~big].sell.mean():.1%}',
      '| доля заказов от 1k+:',f'{K[big].orders.sum()/K.orders.sum():.0%}')
print('резерв: сообщества 1k+ без каталога с живым контентом:',((A.segment=='5 Контент без витрины')&(A.subs>=1000)).sum())

доля сообществ 1k+ среди товарных с каталогом: 44% | доля продающих 1k+ vs <1k: 12.1% vs 2.8% | доля заказов от 1k+: 96%
резерв: сообщества 1k+ без каталога с живым контентом: 871


## 7. Экспорт

In [20]:
b=b.merge(A[['id','segment','stack']],on='id',how='left')
legend=pd.DataFrame({'код':list(COLS)+list(F)+['in_scope','analysis','aov_calc','ads_w','segment','stack'],
   'описание':list(COLS.values())+[F[k][0] for k in F]+['1 = товарный бизнес (не «услужное»)','1 = в аналитической выборке',
   'Средний чек = GMV ÷ заказы','Реклама, обрезанная по 99-му перцентилю','Сегмент (раздел 5)','Уровень инструментов (раздел 3)']})
with pd.ExcelWriter(OUT) as w:
    b.to_excel(w,sheet_name='Бизнесы_с_флагами',index=False); legend.to_excel(w,sheet_name='Легенда',index=False)
    LOGDF.to_excel(w,sheet_name='Лог_очистки',index=False); FUN.to_excel(w,sheet_name='Воронка',index=False); ST.to_excel(w,sheet_name='Уровни_инструментов')
    D.to_excel(w,sheet_name='Драйверы',index=False); CAT.to_excel(w,sheet_name='Размер_каталога'); SG.to_excel(w,sheet_name='Сегменты'); SS.to_excel(w,sheet_name='Сегменты_по_размеру')
    H.to_excel(w,sheet_name='5_гипотез_H1-H3',index=False); M.to_excel(w,sheet_name='H4_Маркет_vs_П1'); CATM.to_excel(w,sheet_name='H4_категории',index=False); H5.to_excel(w,sheet_name='H5_по_размеру',index=False)
print('saved',OUT)

saved ../output/Бизнесы_очищенные.xlsx
